In [1]:
!pip install gradio matplotlib

In [2]:
import random
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

In [3]:
def get_scores(move1, move2):
    if move1 == "Cooperate" and move2 == "Cooperate":
        return 3, 3
    elif move1 == "Betray" and move2 == "Cooperate":
        return 5, 0
    elif move1 == "Cooperate" and move2 == "Betray":
        return 0, 5
    else:
        return 1, 1

In [4]:
def ai_move(strategy, last_opponent_move=None):
    if strategy == "Always Cooperate":
        return "Cooperate"

    elif strategy == "Always Betray":
        return "Betray"

    elif strategy == "Random":
        return random.choice(["Cooperate", "Betray"])

    elif strategy == "Tit-for-Tat":
        # Copy whatever opponent did last round
        if last_opponent_move is None:
            return "Cooperate"  # Be nice on first move
        return last_opponent_move

    elif strategy == "Mostly Cooperate":
        # Cooperate 80% of the time
        return "Cooperate" if random.random() < 0.8 else "Betray"

    elif strategy == "Mostly Betray":
        # Betray 80% of the time
        return "Betray" if random.random() < 0.8 else "Cooperate"

In [5]:
def play_ai_vs_ai(p1_strategy, p2_strategy, rounds):
    score1, score2 = 0, 0
    result = ""
    history1, history2 = [], []
    last_move1, last_move2 = None, None

    for i in range(int(rounds)):
        move1 = ai_move(p1_strategy, last_move2)
        move2 = ai_move(p2_strategy, last_move1)

        points1, points2 = get_scores(move1, move2)
        score1 += points1
        score2 += points2

        history1.append(score1)
        history2.append(score2)

        last_move1, last_move2 = move1, move2

        result += f"Round {i+1}\n"
        result += f"  Player A ({p1_strategy}): {move1}\n"
        result += f"  Player B ({p2_strategy}): {move2}\n"
        result += f"  Scores so far: A={score1}  B={score2}\n\n"

    result += "=" * 30 + "\n"
    result += f"FINAL SCORES\n"
    result += f"  Player A: {score1}\n"
    result += f"  Player B: {score2}\n"

    if score1 > score2:
        result += "🏆 Winner: Player A!"
    elif score2 > score1:
        result += "🏆 Winner: Player B!"
    else:
        result += "🤝 It's a Tie!"

    # Draw chart
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, int(rounds)+1), history1, label=f"Player A ({p1_strategy})", color="blue", linewidth=2)
    ax.plot(range(1, int(rounds)+1), history2, label=f"Player B ({p2_strategy})", color="red", linewidth=2)
    ax.set_title("Score Progression Over Rounds")
    ax.set_xlabel("Round")
    ax.set_ylabel("Total Score")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    return result, fig

In [6]:
def play_human_vs_ai(human_move, ai_strategy, rounds):
    score_human, score_ai = 0, 0
    result = ""
    history_human, history_ai = [], []
    last_human_move = None

    for i in range(int(rounds)):
        ai_choice = ai_move(ai_strategy, last_human_move)
        points_human, points_ai = get_scores(human_move, ai_choice)
        score_human += points_human
        score_ai += points_ai

        history_human.append(score_human)
        history_ai.append(score_ai)

        last_human_move = human_move

        result += f"Round {i+1}\n"
        result += f"  You: {human_move}\n"
        result += f"  AI ({ai_strategy}): {ai_choice}\n"
        result += f"  Scores so far: You={score_human}  AI={score_ai}\n\n"

    result += "=" * 30 + "\n"
    result += f"FINAL SCORES\n"
    result += f"  You: {score_human}\n"
    result += f"  AI:  {score_ai}\n"

    if score_human > score_ai:
        result += "🏆 You Win!"
    elif score_ai > score_human:
        result += "🤖 AI Wins!"
    else:
        result += "🤝 It's a Tie!"

    # Draw chart
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, int(rounds)+1), history_human, label="You", color="green", linewidth=2)
    ax.plot(range(1, int(rounds)+1), history_ai, label=f"AI ({ai_strategy})", color="red", linewidth=2)
    ax.set_title("Your Score vs AI Score")
    ax.set_xlabel("Round")
    ax.set_ylabel("Total Score")
    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    return result, fig

In [7]:
def play_tournament(rounds):
    strategies = [
        "Always Cooperate",
        "Always Betray",
        "Random",
        "Tit-for-Tat",
        "Mostly Cooperate",
        "Mostly Betray"
    ]

    total_scores = {s: 0 for s in strategies}
    result = "🏟️ TOURNAMENT RESULTS\n"
    result += "Every strategy fights every other strategy!\n\n"

    for i, s1 in enumerate(strategies):
        for j, s2 in enumerate(strategies):
            if i >= j:
                continue
            sc1, sc2 = 0, 0
            last1, last2 = None, None
            for _ in range(int(rounds)):
                m1 = ai_move(s1, last2)
                m2 = ai_move(s2, last1)
                p1, p2 = get_scores(m1, m2)
                sc1 += p1
                sc2 += p2
                last1, last2 = m1, m2
            total_scores[s1] += sc1
            total_scores[s2] += sc2
            result += f"{s1} vs {s2} → {sc1} : {sc2}\n"

    # Ranking
    ranked = sorted(total_scores.items(), key=lambda x: x[1], reverse=True)
    result += "\n" + "=" * 30 + "\n"
    result += "🥇 FINAL RANKINGS\n"
    medals = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣", "6️⃣"]
    for idx, (strategy, score) in enumerate(ranked):
        result += f"{medals[idx]} {strategy}: {score} points\n"

    # Bar chart
    names = [r[0] for r in ranked]
    scores = [r[1] for r in ranked]
    colors = ["gold", "silver", "#cd7f32", "skyblue", "lightgreen", "salmon"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, scores, color=colors, edgecolor="black")
    ax.set_title("🏆 Tournament Final Scores", fontsize=14)
    ax.set_xlabel("Strategy")
    ax.set_ylabel("Total Points")
    ax.set_xticklabels(names, rotation=20, ha='right')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(score), ha='center', fontsize=10, fontweight='bold')
    plt.tight_layout()

    return result, fig

In [8]:
strategies_list = [
    "Always Cooperate",
    "Always Betray",
    "Random",
    "Tit-for-Tat",
    "Mostly Cooperate",
    "Mostly Betray"
]

with gr.Blocks(title="Prisoner's Dilemma Simulator") as app:

    gr.Markdown("# 🧠 Prisoner's Dilemma Strategy Simulator")
    gr.Markdown("Explore game theory with AI strategies, human play, and a full tournament!")

    # --- TAB 1: AI vs AI ---
    with gr.Tab("🤖 AI vs AI"):
        gr.Markdown("### Two AI strategies compete against each other")
        with gr.Row():
            p1 = gr.Dropdown(strategies_list, value="Tit-for-Tat", label="Player A Strategy")
            p2 = gr.Dropdown(strategies_list, value="Always Betray", label="Player B Strategy")
        rounds1 = gr.Slider(1, 50, value=10, step=1, label="Number of Rounds")
        btn1 = gr.Button("▶ Run Game", variant="primary")
        out1_text = gr.Textbox(label="Round-by-Round Results", lines=25, max_lines=60)
        out1_chart = gr.Plot(label="Score Chart")
        btn1.click(fn=play_ai_vs_ai, inputs=[p1, p2, rounds1], outputs=[out1_text, out1_chart])

    # --- TAB 2: Human vs AI ---
    with gr.Tab("🧑 Human vs AI"):
        gr.Markdown("### You pick your move — AI plays its strategy every round")
        with gr.Row():
            human_move = gr.Radio(["Cooperate", "Betray"], label="Your Move", value="Cooperate")
            ai_strat = gr.Dropdown(strategies_list, value="Tit-for-Tat", label="AI Strategy")
        rounds2 = gr.Slider(1, 50, value=10, step=1, label="Number of Rounds")
        btn2 = gr.Button("▶ Play Game", variant="primary")
        out2_text = gr.Textbox(label="Results", lines=25, max_lines=60)
        out2_chart = gr.Plot(label="Score Chart")
        btn2.click(fn=play_human_vs_ai, inputs=[human_move, ai_strat, rounds2], outputs=[out2_text, out2_chart])

    # --- TAB 3: Tournament ---
    with gr.Tab("🏆 Tournament"):
        gr.Markdown("### All 6 strategies fight each other — who wins overall?")
        rounds3 = gr.Slider(1, 50, value=15, step=1, label="Rounds per Match")
        btn3 = gr.Button("▶ Start Tournament", variant="primary")
        out3_text = gr.Textbox(label="Tournament Results", lines=25, max_lines=60)
        out3_chart = gr.Plot(label="Final Rankings Chart")
        btn3.click(fn=play_tournament, inputs=[rounds3], outputs=[out3_text, out3_chart])

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3647b93ee88d3ff62a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
